# Bank Customer Churn Preprocessing

This notebook prepares the Bank Customer Churn dataset for downstream modeling. The workflow includes handling missing values, removing unnecessary columns, encoding categorical features, splitting the data into training and testing sets, scaling features, and saving the processed data.

## 1. Import Libraries

We begin by importing the libraries required for data handling, preprocessing, and model preparation.

In [2]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Dataset

The raw dataset is loaded from the datasets folder. A fallback path is included to make the notebook robust when run from different directories.

In [3]:
# Locate the raw dataset
raw_path = Path("../datasets/raw/Churn_Modelling.csv")
if not raw_path.exists():
    raw_path = Path("../../datasets/raw/Churn_Modelling.csv")

if not raw_path.exists():
    raise FileNotFoundError(f"Dataset not found at expected locations: {raw_path}")

# Load the dataset
df = pd.read_csv(raw_path)
print(f"Dataset loaded successfully from: {raw_path}")
print("\nShape of the dataset:", df.shape)
df.head()

Dataset loaded successfully from: ..\..\datasets\raw\Churn_Modelling.csv

Shape of the dataset: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 3. Inspect the Data

Before preprocessing, we review the columns and the current data types to decide which features should be retained or transformed.

In [4]:
# Review column names and data types
print(df.dtypes)
print("\nColumns in the dataset:")
print(df.columns.tolist())

RowNumber            int64
CustomerId           int64
Surname                str
CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

Columns in the dataset:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


## 4. Handle Missing Values

If missing values are present, we fill them using simple and safe strategies. Numeric columns are filled with their median, while categorical columns use the mode.

In [5]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values before preprocessing:")
print(missing_values[missing_values > 0])

# Handle missing values if present
if missing_values.sum() > 0:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns

    for col in numeric_cols:
        df[col].fillna(df[col].median(), inplace=True)

    for col in categorical_cols:
        df[col].fillna(df[col].mode()[0], inplace=True)

    print("\nMissing values after preprocessing:")
    print(df.isnull().sum().sum())
else:
    print("No missing values detected.")

Missing values before preprocessing:
Series([], dtype: int64)
No missing values detected.


## 5. Remove Unnecessary Columns

Some columns such as identifiers or text fields are not useful for modeling and should be removed before training.

In [6]:
# Remove non-informative identifier-like columns
columns_to_drop = ["RowNumber", "CustomerId", "Surname"]

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print("Remaining columns:")
print(df.columns.tolist())

Remaining columns:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


## 6. Encode Categorical Features

Categorical features are converted into numeric form so the machine learning model can process them. Here, we use one-hot encoding for the categorical columns.

In [7]:
# Identify categorical columns after dropping unnecessary columns
categorical_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Categorical columns to encode:", categorical_columns)

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

print("\nEncoded dataset shape:", df_encoded.shape)
df_encoded.head()

Categorical columns to encode: ['Geography', 'Gender']

Encoded dataset shape: (10000, 12)


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False


## 7. Split Features and Target

The target variable is Exited. We separate it from the feature matrix and prepare the data for train-test splitting.

In [8]:
# Split into features and target
X = df_encoded.drop(columns=["Exited"])
y = df_encoded["Exited"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (10000, 11)
Target vector shape: (10000,)


## 8. Train-Test Split

The data is split into training and testing sets so the model can be evaluated on unseen data.

In [9]:
# Create train and test splits
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (8000, 11)
Testing set shape: (2000, 11)


## 9. Feature Scaling

Feature scaling standardizes the numeric values so that features with larger ranges do not dominate the learning process.

In [10]:
# Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames for easier handling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Feature scaling completed.")
X_train_scaled_df.head()

Feature scaling completed.


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
2151,1.058568,1.715086,0.684723,-1.226059,-0.910256,0.641042,-1.030206,1.042084,-0.578313,-0.577735,0.907507
8392,0.913626,-0.659935,-0.696202,0.413288,-0.910256,0.641042,-1.030206,-0.623556,1.729169,-0.577735,0.907507
5006,1.079274,-0.184931,-1.731895,0.601687,0.808830,0.641042,0.970680,0.308128,1.729169,-0.577735,-1.101919
4117,-0.929207,-0.184931,-0.005739,-1.226059,0.808830,0.641042,-1.030206,-0.290199,-0.578313,-0.577735,0.907507
7182,0.427035,0.955079,0.339492,0.548318,0.808830,-1.559960,0.970680,0.135042,1.729169,-0.577735,0.907507


## 10. Save the Processed Dataset

The processed training and testing data are saved to the processed folder for future use.

In [11]:
# Create output directory if needed
output_dir = Path("../datasets/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save processed datasets
X_train_scaled_df.to_csv(output_dir / "X_train_scaled.csv", index=False)
X_test_scaled_df.to_csv(output_dir / "X_test_scaled.csv", index=False)
y_train.to_csv(output_dir / "y_train.csv", index=False, header=True)
y_test.to_csv(output_dir / "y_test.csv", index=False, header=True)

# Save a combined processed dataset as well
processed_df = pd.concat([X_train_scaled_df, y_train.rename("Exited")], axis=1)
processed_df.to_csv(output_dir / "processed_dataset.csv", index=False)

print("Processed datasets saved successfully to:", output_dir)
print("Files created:")
for file_path in sorted(output_dir.glob("*")):
    print("-", file_path.name)

Processed datasets saved successfully to: ..\datasets\processed
Files created:
- processed_dataset.csv
- X_test_scaled.csv
- X_train_scaled.csv
- y_test.csv
- y_train.csv


In [12]:
import joblib
from pathlib import Path

# Resolve the backend folder from the current notebook working directory
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
backend_root = next((root / "backend" for root in candidate_roots if (root / "backend").exists()), None)
if backend_root is None:
    backend_root = Path.cwd()

processed_dir = backend_root / "datasets" / "processed"
saved_models_dir = backend_root / "saved_models"
processed_dir.mkdir(parents=True, exist_ok=True)
saved_models_dir.mkdir(parents=True, exist_ok=True)

# Save train/test splits and targets for the FastAPI backend
X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)
y_train.to_frame(name="Exited").to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_frame(name="Exited").to_csv(processed_dir / "y_test.csv", index=False)

# Save the fitted scaler and encoder metadata
scaler_path = saved_models_dir / "scaler.pkl"
encoder_path = saved_models_dir / "label_encoders.pkl"

joblib.dump(scaler, scaler_path)
joblib.dump({"encoding_type": "one_hot", "categorical_columns": categorical_columns}, encoder_path)

print("Deployment artifacts saved successfully.")
print("- Saved:", scaler_path.resolve())
print("- Saved:", encoder_path.resolve())
print("- Saved:", (processed_dir / "X_train.csv").resolve())
print("- Saved:", (processed_dir / "X_test.csv").resolve())
print("- Saved:", (processed_dir / "y_train.csv").resolve())
print("- Saved:", (processed_dir / "y_test.csv").resolve())
print("- Saved:", saved_models_dir / "label_encoders.pkl")

Deployment artifacts saved successfully.
- Saved: C:\Users\MITHRA\RetainIQ\backend\saved_models\scaler.pkl
- Saved: C:\Users\MITHRA\RetainIQ\backend\saved_models\label_encoders.pkl
- Saved: C:\Users\MITHRA\RetainIQ\backend\datasets\processed\X_train.csv
- Saved: C:\Users\MITHRA\RetainIQ\backend\datasets\processed\X_test.csv
- Saved: C:\Users\MITHRA\RetainIQ\backend\datasets\processed\y_train.csv
- Saved: C:\Users\MITHRA\RetainIQ\backend\datasets\processed\y_test.csv
- Saved: C:\Users\MITHRA\RetainIQ\backend\saved_models\label_encoders.pkl
